In [1]:
import pandas as pd
import numpy as np
from catnip.fla_redshift import FLA_Redshift
from sqlalchemy import null
from datetime import datetime

from prefect.blocks.system import Secret
from typing import Dict
from concurrent.futures import ThreadPoolExecutor

In [2]:
def get_redshift_credentials() -> Dict:

    cred_dict = {
        "dbname": Secret.load("stellar-redshift-db-name").get(),
        "host": Secret.load("stellar-redshift-host").get(),
        "port": 5439,
        "user": Secret.load("stellar-redshift-user-name").get(),
        "password": Secret.load("stellar-redshift-password").get(),

        "aws_access_key_id": Secret.load("fla-s3-aws-access-key-id-east-1").get(),
        "aws_secret_access_key": Secret.load("fla-s3-aws-secret-access-key-east-1").get(),
        "bucket": Secret.load("fla-s3-bucket-name-east-1").get(),
        "subdirectory": "us-east-1",

        "verbose": False,
    }

    return cred_dict

with ThreadPoolExecutor(1) as pool:
    rs_creds = pool.submit(lambda: get_redshift_credentials()).result()

In [ ]:
# what is the cutoff date year to year?
# do we want to group programs by larger type or location?
# what is the goal of managing the data?

# try hockey for free has no times (first tab)
# school raffles and PE visits blank

In [ ]:
# 1. combine all programs
# 2. just have email and join date
# 3. have function for calculating season
# 4. 

In [ ]:
df = pd.read_csv("C:\\Users\\riffere\\OneDrive - Florida Panthers\\Documents\\LTP\\LTP_Combined.csv")
FLA_Redshift(**rs_creds).write_to_warehouse(df = df, table_name= "community_ltp_data")

In [ ]:
def update_programs(df, type, year):

    q = """
    WITH initial AS (
    SELECT
        LOWER(parent_email) AS parent_email,
        MIN(season) AS season,
        MIN(date(transaction_date)) AS transaction_date
    FROM custom.community_ltp_data
    GROUP BY parent_email
),
acct_rev_info AS (
    SELECT
        c.email,
        t.season,
        t.createdon AS add_date,
        t.gross_revenue,
        CASE
            WHEN t.ticket_type IN ('Annual Suites', 'Full', 'Half', 'Premier') THEN 1
            ELSE 0
        END AS is_stm
    FROM custom.cth_v_historical_ticket t
    JOIN custom.korepss_externalsystemtocontact ext
        ON t.purchaser_ticketing_id = ext.externalcontactid
    JOIN custom.korepss_contacts c
        ON ext.crmcontactid = c.contactid
),
subs_2526 AS (
    SELECT
        c.email,
        '2025-26' AS season,
        SUM(subs.gross_revenue) AS gross_revenue,
        1 AS is_stm
    FROM custom.cth_v_ticket_subscription_2526 subs
    JOIN custom.korepss_externalsystemtocontact ext
        ON subs.purch_client_crm_id = ext.externalcontactid
    JOIN custom.korepss_contacts c
        ON ext.crmcontactid = c.contactid
    GROUP BY c.email
),
acct_rev_all AS (
    SELECT email, season,
           MIN(add_date) AS add_date,
           SUM(gross_revenue) AS gross_revenue,
           MAX(is_stm) AS is_stm
    FROM (
        SELECT email, season, add_date, gross_revenue, is_stm
        FROM acct_rev_info
        UNION ALL
        SELECT email, season, NULL AS add_date, gross_revenue, is_stm
        FROM subs_2526
    ) x
    GROUP BY email, season
)
SELECT
    i.parent_email,
    i.transaction_date,
    i.season,
    a.add_date,
    a.gross_revenue,
    a.is_stm
FROM initial i
LEFT JOIN acct_rev_all a
    ON i.parent_email = a.email
   AND i.season = a.season
WHERE i.parent_email IS NOT NULL
  AND i.parent_email <> ''
  AND i.season IN ('2021-22','2022-23','2023-24','2024-25','2025-26');


"""
     

    return